# Material Behavior Prediction Using Machine Learning
## End-to-End Educational Portfolio Notebook

### Problem Overview:
In materials science and structural engineering, determining how strong a concrete formulation will be traditionally requires mixing batches, casting specimens, curing them under controlled conditions for days, weeks, or months, and then destructively crushing them in hydraulic test frames.

$$\text{Material Composition} + \text{Curing Conditions (Age)} \xrightarrow{\text{ML Model}} \text{Predicted Mechanical Strength (MPa)}$$

This notebook guides you through the complete end-to-end machine learning lifecycle from raw data to trained model deployment.

## 1. Setup & Environment Dependencies

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
print("Libraries successfully imported!")

## 2. Milestone 1: Data Understanding & Hygiene Inspection

In [ ]:
# Load dataset from local data directory
data_path = os.path.join('..', 'data', 'material_data.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'material_data.csv')

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
df.head()

In [ ]:
# Statistical summary
df.describe().round(2)

## 3. Milestone 2: Exploratory Data Analysis (EDA)
Let's visualize the target variable distribution and the fundamental materials science relationship: **Abrams' Law** (Water-to-Cement Ratio vs. Compressive Strength).

In [ ]:
# Plot Target Distribution
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={'height_ratios': [0.25, 0.75]})
sns.boxplot(x=df['strength'], ax=ax_box, color='#60a5fa')
sns.histplot(df['strength'], kde=True, ax=ax_hist, color='#2563eb', bins=25)
ax_hist.set_xlabel("Compressive Strength (MPa)", fontweight='bold')
plt.suptitle("Target Distribution: Concrete Compressive Strength", fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# Validating Abrams' Law: Water-to-Cement Ratio vs. Strength
w_c = df['water'] / df['cement']
plt.figure(figsize=(8, 5))
scatter = plt.scatter(w_c, df['strength'], c=df['age'], cmap='viridis', alpha=0.75)
plt.colorbar(scatter, label='Curing Age (Days)')
plt.title("Verification of Abrams' Law: Water-to-Cement Ratio vs. Strength", fontweight='bold')
plt.xlabel("Water-to-Cement Ratio (w/c)", fontweight='bold')
plt.ylabel("Compressive Strength (MPa)", fontweight='bold')
plt.show()

## 4. Milestone 3: Preprocessing & Data Leakage Prevention
- Drop exact duplicate entries (preventing test set memorization).
- 80/20 train/test split with `random_state=42`.
- Standardize features strictly fitting scaler on `X_train` only.

In [ ]:
# 1. Drop duplicates
df_clean = df.drop_duplicates().copy()
print(f"Samples after dropping duplicates: {len(df_clean)} (dropped {len(df) - len(df_clean)})")

# 2. Separate X and y
X = df_clean.drop(columns=['strength'])
y = df_clean['strength']

# 3. Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape[0]} samples | Testing set: {X_test.shape[0]} samples")

# 4. Feature Standardization (Anti-leakage: fit on train only!)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
print("Features scaled successfully!")

## 5. Milestone 4: Baseline Linear Regression

In [ ]:
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)

lr_test_preds = baseline_lr.predict(X_test_scaled)
lr_r2 = r2_score(y_test, lr_test_preds)
lr_mae = mean_absolute_error(y_test, lr_test_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_test_preds))

print(f"Baseline Linear Regression Test R²  : {lr_r2:.4f}")
print(f"Baseline Linear Regression Test MAE : {lr_mae:.2f} MPa")
print(f"Baseline Linear Regression Test RMSE: {lr_rmse:.2f} MPa")

## 6. Milestone 5: Multi-Model Benchmark Comparison

In [ ]:
models = {
    'Linear Regression': (LinearRegression(), X_train_scaled, X_test_scaled),
    'Decision Tree': (DecisionTreeRegressor(max_depth=6, random_state=42), X_train, X_test),
    'Random Forest': (RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42), X_train, X_test),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42), X_train, X_test)
}

results = []
for name, (mod, xtr, xte) in models.items():
    mod.fit(xtr, y_train)
    tr_preds = mod.predict(xtr)
    te_preds = mod.predict(xte)
    results.append({
        'Model': name,
        'Train R²': round(r2_score(y_train, tr_preds), 4),
        'Test R²': round(r2_score(y_test, te_preds), 4),
        'Test MAE (MPa)': round(mean_absolute_error(y_test, te_preds), 2),
        'Test RMSE (MPa)': round(np.sqrt(mean_squared_error(y_test, te_preds)), 2)
    })

comparison_df = pd.DataFrame(results)
comparison_df

## 7. Milestone 6: Model Interpretation & Feature Importance
Let's inspect what features drive the predictions of our champion model (Gradient Boosting).

In [ ]:
gb_model = models['Gradient Boosting'][0]
perm = permutation_importance(gb_model, X_test, y_test, n_repeats=10, random_state=42)

imp_df = pd.DataFrame({
    'Feature': X.columns,
    'MDI_Importance': gb_model.feature_importances_,
    'Test_Permutation_Drop': perm.importances_mean
}).sort_values(by='Test_Permutation_Drop', ascending=False)

imp_df

## 8. Milestone 7: Actual vs. Predicted & Error Analysis

In [ ]:
gb_preds = gb_model.predict(X_test)
residuals = y_test - gb_preds

plt.figure(figsize=(7, 7))
plt.scatter(y_test, gb_preds, color='#0284c7', alpha=0.75)
plt.plot([0, 85], [0, 85], color='red', linestyle='--')
plt.fill_between([0, 85], [-5, 80], [5, 90], color='green', alpha=0.15, label='±5 MPa Tolerance')
plt.title("Actual vs. Predicted Strength (Gradient Boosting)", fontweight='bold')
plt.xlabel("Actual Strength (MPa)", fontweight='bold')
plt.ylabel("Predicted Strength (MPa)", fontweight='bold')
plt.legend()
plt.show()

## 9. Milestone 8: Predicting a New Material Formulation
Use our trained model to predict the behavior of any custom mix design!

In [ ]:
# Define a new mix: Sustainable Green Concrete with Slag & Ash
new_mix = pd.DataFrame([{
    'cement': 220.0,       # kg/m³
    'slag': 110.0,         # kg/m³
    'ash': 75.0,           # kg/m³
    'water': 155.0,        # kg/m³
    'superplastic': 9.0,   # kg/m³
    'coarseagg': 960.0,    # kg/m³
    'fineagg': 740.0,      # kg/m³
    'age': 28              # days
}])

predicted_mpa = gb_model.predict(new_mix)[0]
w_b = new_mix['water'][0] / (new_mix['cement'][0] + new_mix['slag'][0] + new_mix['ash'][0])

print(f"Input Formulation Water-to-Binder Ratio: {w_b:.2f}")
print(f"Estimated Compressive Strength         : {predicted_mpa:.2f} MPa")